# AutoMode rebuttal — tables and figures from verified runs

Reads the result download made by `AutoMode_Four_GPU_Colab.ipynb` (section 10). Rules:

* Only ledger rows with `status == "completed"` are used; each is re-checked against its own `result.json`.
* Runs are grouped by the exact **(variant, train_track, protocol_version)**; nothing is pooled across variants or protocols, and no pre-correction outputs are read.
* Comparisons are **paired by seed** (and per item by `item_id`), never by array position.
* With three seeds the **seed SD is descriptive**; no significance tests are computed.
* Layer occupancy is **weighted by optimizer updates** (from `switch_diagnostics.json`), not averaged over irregular snapshots; the initial allocation is reported separately.
* Trainable-parameter counts are **not** measured compute; measured wall time and memory are shown separately.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from automode.analysis import summarize_runs, paired_seed_differences, paired_item_table

DOWNLOAD_ROOT = Path('PATH/TO/automode_rebuttal_control/CAMPAIGN_COMMIT/downloads/TIMESTAMP')
FIG_DIR = Path('rebuttal_figures'); FIG_DIR.mkdir(exist_ok=True)
METRIC = 'eval_gsm8k_maj3'

rows, run_dirs, identities = [], {}, set()
for ledger_path in sorted(DOWNLOAD_ROOT.glob('worker*/results/run_ledger.json')):
    ledger = json.loads(ledger_path.read_text())
    identities.add((ledger['commit'], ledger['manifest_sha256']))
    for r in ledger['runs']:
        if r['status'] != 'completed':
            print(f"not completed: {r['job_id']} ({r['status']})")
            continue
        run_dir = ledger_path.parent / r['result_dir']
        res = json.loads((run_dir / 'result.json').read_text())
        assert res['status'] == 'completed' and res[METRIC] == r[METRIC] and res['eval_gsm8k_n_total'] == 1319
        rows.append({'job_id': r['job_id'], 'variant': res['variant'], 'method': res['method'],
                     'train_track': res['train_track'], 'protocol_version': res['protocol_version'],
                     'seed': res['seed'], METRIC: res[METRIC],
                     'train_time_h': res['train_time_sec'] / 3600, 'eval_time_h': res['eval_time_sec'] / 3600,
                     'peak_vram_reserved_gib': res.get('peak_vram_reserved_gib'),
                     'avg_trainable_params_M': res['train']['avg_trainable_params_step_weighted'] / 1e6,
                     'switch_overhead_s': res['train']['switch_overhead_sec']})
        run_dirs[r['job_id']] = run_dir
assert len(identities) <= 1, f'ledgers from different commits/manifests: {identities}'
assert len({r['job_id'] for r in rows}) == len(rows)
df = pd.DataFrame(rows)
print(f'{len(df)} verified completed runs'); df

In [ ]:
# ── Main table: mean and seed SD per exact variant (descriptive, n=3 seeds) ──
summary = pd.DataFrame(summarize_runs(rows, METRIC))
summary['mean_%'] = 100 * summary['mean']
summary['seed_sd_%'] = 100 * summary['seed_sd']
print(summary[['variant', 'train_track', 'protocol_version', 'n_seeds', 'seeds', 'mean_%', 'seed_sd_%']].to_string(index=False))
summary.to_csv(FIG_DIR / 'main_table.csv', index=False)

auto = next(v for v in df.variant.unique() if v.startswith('automode'))
for other in sorted(set(df.variant) - {auto}):
    track, proto = df.train_track.iloc[0], df.protocol_version.iloc[0]
    d = paired_seed_differences(rows, METRIC, auto, other, track, proto)
    per_seed = {s: round(100 * v, 2) for s, v in d['diff_by_seed'].items()}
    print(f"{auto} - {other}: per-seed {per_seed} "
          f"mean {None if d['mean_diff'] is None else round(100*d['mean_diff'], 2)} pts "
          f"(seed SD of diff {None if d['seed_sd_of_diff'] is None else round(100*d['seed_sd_of_diff'], 2)}); "
          f"unpaired seeds {d['unpaired_seeds']}")

In [ ]:
# ── Item-level pairing by item_id within each seed ─────────────────────────
def records(job_id):
    return json.loads((run_dirs[job_id] / 'evals' / 'gsm8k_eval.json').read_text())['records']

item_rows = []
for seed in sorted(df.seed.unique()):
    a = df[(df.variant == auto) & (df.seed == seed)]
    for _, b in df[(df.variant != auto) & (df.seed == seed)].iterrows():
        if len(a):
            item_rows.append({'seed': seed, 'other': b.variant,
                              **paired_item_table(records(a.job_id.iloc[0]), records(b.job_id))})
pd.DataFrame(item_rows)

In [ ]:
# ── Figure: every seed shown, mean as a bar ────────────────────────────────
order = list(summary.sort_values('mean').variant)
fig, ax = plt.subplots(figsize=(8, 3.8))
for i, v in enumerate(order):
    sub = df[df.variant == v]
    ax.bar(i, 100 * sub[METRIC].mean(), color='0.85', edgecolor='black', linewidth=0.6)
    ax.scatter([i] * len(sub), 100 * sub[METRIC], color='black', s=14, zorder=3)
    for _, r in sub.iterrows():
        ax.annotate(str(r.seed), (i, 100 * r[METRIC]), xytext=(4, 0), textcoords='offset points', fontsize=7)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('GSM8K maj@3 (%)'); ax.set_title('Per-seed results (bar = mean over seeds; no significance implied)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'results_per_seed.pdf'); plt.show()

In [ ]:
# ── Per-run training trajectories and actual LR schedules ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for _, r in df.sort_values(['variant', 'seed']).iterrows():
    loss = json.loads((run_dirs[r.job_id] / 'logs' / 'training_loss.json').read_text())['loss_log']
    lr = json.loads((run_dirs[r.job_id] / 'logs' / 'lr_schedule.json').read_text())['lr_log']
    axes[0].plot([e['step'] for e in loss], [e['loss'] for e in loss], lw=0.8, label=f'{r.variant} s{r.seed}')
    axes[1].plot([e['step'] for e in lr], [e['lr'] for e in lr], lw=0.8)
axes[0].set_xlabel('optimizer update'); axes[0].set_ylabel('train loss (window mean)')
axes[1].set_xlabel('optimizer update'); axes[1].set_ylabel('learning rate used')
axes[0].legend(fontsize=5, ncol=2)
plt.tight_layout(); plt.savefig(FIG_DIR / 'trajectories.pdf'); plt.show()
lr_seqs = {tuple(e['lr'] for e in json.loads((run_dirs[j] / 'logs' / 'lr_schedule.json').read_text())['lr_log'])
           for j in df.job_id}
print('distinct LR sequences across all runs:', len(lr_seqs), '(1 expected: switching must not change the schedule)')

In [ ]:
# ── Switch diagnostics: permanence, re-promotion, update-weighted occupancy ─
diag_rows = []
for _, r in df[df.method.isin(['automode', 'select_once', 'fixed_mixed', 'static_qv'])].iterrows():
    d = json.loads((run_dirs[r.job_id] / 'dynamic' / 'switch_diagnostics.json').read_text())
    perm = d['first_selected_remaining_base_rest_of_training'] or {}
    diag_rows.append({'job_id': r.job_id, 'variant': r.variant, 'seed': r.seed,
                      'decision_events': d['n_decision_events'], 'initial_base_layers': d['initial_modes_count']['full_ft'],
                      'first_selection_size': (d['first_selection'] or {}).get('size'),
                      'first_selected_stayed_base': f"{perm.get('count')}/{perm.get('denominator')}",
                      'promotions': d['totals']['promotions'], 'demotions': d['totals']['demotions'],
                      're_promotions': d['totals']['re_promotions'], 're_promotion_note': d['re_promotion']['note'],
                      'mean_successive_jaccard': d['mean_successive_jaccard']['value'],
                      'mean_base_occupancy': d['mean_base_occupancy_fraction'],
                      'step_weighted_trainable_M': d['optimizer_step_weighted_trainable_params'] / 1e6})
    occ = [d['per_layer'][f'layer_{i}']['base_occupancy_fraction'] for i in range(d['n_layers'])]
    if r.method in ('automode', 'select_once'):
        fig, ax = plt.subplots(figsize=(9, 1.4))
        im = ax.imshow([occ], aspect='auto', cmap='Greys', vmin=0, vmax=1)
        ax.set_yticks([]); ax.set_xlabel('layer'); ax.set_title(f'{r.variant} seed {r.seed}: fraction of updates in base mode')
        plt.colorbar(im, ax=ax, fraction=0.03); plt.tight_layout()
        plt.savefig(FIG_DIR / f'occupancy_{r.job_id}.pdf'); plt.show()
pd.DataFrame(diag_rows)

In [ ]:
# ── Measured cost (separate from parameter counts) ─────────────────────────
cost = df.groupby('variant')[['train_time_h', 'eval_time_h', 'peak_vram_reserved_gib',
                              'avg_trainable_params_M', 'switch_overhead_s']].agg(['mean', 'min', 'max'])
print('Trainable-parameter counts are a capacity measure, not measured compute.')
cost